## Load Model from DagsHub Registry
- In this section I connect to the MLflow Model Registry on DagsHub to retrieve the "Champion" model. By using the latest version of the XGBoost model I ensure that the inference is performed using the most optimized and updated version from our experiments.

In [53]:
import pandas as pd
import numpy as np
import mlflow
import dagshub
import os

# Setup MLflow and DagsHub connection
os.environ['MLFLOW_TRACKING_USERNAME'] = 'GigiSichinava'
dagshub.init(repo_owner='GigiSichinava', repo_name='ML-Assignment-1')
mlflow.set_tracking_uri("https://dagshub.com/GigiSichinava/ML-Assignment-1.mlflow")

print("DagsHub connection established.")

Initialized MLflow to track repo "GigiSichinava/ML-Assignment-1"

Repository GigiSichinava/ML-Assignment-1 initialized!

DagsHub connection established.


## Test Data Preparation
To ensure the model functions correctly the test features must match the **exact preprocessing** used during training.

This inference notebook therefore reuses the same pipeline as the experiment notebook:
- Smart NaN handling
- Feature creation
- Ordinal encoding for ordered quality/condition columns
- One-hot encoding for nominal categoricals (Train-Test combined so columns align)
- Optional correlation-based feature filtering

In [54]:
# Load data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')


def preprocess_features(train_df: pd.DataFrame, test_df: pd.DataFrame, *, corr_threshold=0.01):
    train = train_df.copy()
    test = test_df.copy()

    y = train["SalePrice"].copy() if "SalePrice" in train.columns else None
    train_features = train.drop(columns=["SalePrice"], errors="ignore")

    # Combine for consistent encoding
    all_features = pd.concat([train_features, test], axis=0, ignore_index=True)

    # Smart NaN handling
    none_like_cats = [
        "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
        "GarageType", "GarageFinish", "GarageQual", "GarageCond",
        "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
        "MasVnrType",
    ]
    for c in none_like_cats:
        if c in all_features.columns:
            all_features[c] = all_features[c].astype("object").fillna("None")

    # LotFrontage: neighborhood-wise median (fallback to global median)
    if "LotFrontage" in all_features.columns:
        if "Neighborhood" in all_features.columns:
            lf = pd.to_numeric(all_features["LotFrontage"], errors="coerce")
            nb = all_features["Neighborhood"].astype("object")
            all_features["LotFrontage"] = lf.fillna(lf.groupby(nb).transform("median"))
        all_features["LotFrontage"] = pd.to_numeric(all_features["LotFrontage"], errors="coerce").fillna(
            pd.to_numeric(all_features["LotFrontage"], errors="coerce").median()
        )

    # Numeric columns where NA generally means "does not exist" -> 0
    zero_fill_nums = [
        "GarageArea", "GarageCars",
        "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
        "BsmtFullBath", "BsmtHalfBath",
        "MasVnrArea",
    ]
    for c in zero_fill_nums:
        if c in all_features.columns:
            all_features[c] = pd.to_numeric(all_features[c], errors="coerce").fillna(0)

    # GarageYrBlt: 0 if no garage, else median
    if "GarageYrBlt" in all_features.columns:
        gy = pd.to_numeric(all_features["GarageYrBlt"], errors="coerce")
        if "GarageType" in all_features.columns:
            no_garage = all_features["GarageType"].astype("object").eq("None")
            gy = gy.fillna(gy.median())
            gy = gy.mask(no_garage, 0)
        else:
            gy = gy.fillna(gy.median())
        all_features["GarageYrBlt"] = gy

    # Remaining numeric NaNs -> median
    num_cols = all_features.select_dtypes(include=[np.number]).columns
    for c in num_cols:
        if all_features[c].isna().any():
            all_features[c] = all_features[c].fillna(all_features[c].median())

    # Remaining categorical NaNs -> mode (fallback "None")
    obj_cols = all_features.select_dtypes(include=["object"]).columns
    for c in obj_cols:
        if all_features[c].isna().any():
            mode = all_features[c].mode(dropna=True)
            all_features[c] = all_features[c].fillna(mode.iloc[0] if len(mode) else "None")

    # Feature creation
    if {"TotalBsmtSF", "1stFlrSF", "2ndFlrSF"}.issubset(all_features.columns):
        all_features["TotalSF"] = all_features["TotalBsmtSF"] + all_features["1stFlrSF"] + all_features["2ndFlrSF"]

    bath_cols = ["FullBath", "HalfBath", "BsmtFullBath", "BsmtHalfBath"]
    if set(bath_cols).issubset(all_features.columns):
        all_features["TotalBathrooms"] = (
            all_features["FullBath"]
            + 0.5 * all_features["HalfBath"]
            + all_features["BsmtFullBath"]
            + 0.5 * all_features["BsmtHalfBath"]
        )

    porch_cols = ["OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch"]
    if set(porch_cols).issubset(all_features.columns):
        all_features["TotalPorchSF"] = (
            all_features["OpenPorchSF"]
            + all_features["EnclosedPorch"]
            + all_features["3SsnPorch"]
            + all_features["ScreenPorch"]
        )

    # Ordinal encoding
    qual_map = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
    exposure_map = {"None": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4}
    finish_map = {"None": 0, "Unf": 1, "RFn": 2, "Fin": 3}
    fin_type_map = {"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6}
    functional_map = {"Sal": 1, "Sev": 2, "Maj2": 3, "Maj1": 4, "Mod": 5, "Min2": 6, "Min1": 7, "Typ": 8}
    paved_map = {"N": 0, "P": 1, "Y": 2}

    ordinal_cols = {
        "ExterQual": qual_map,
        "ExterCond": qual_map,
        "BsmtQual": qual_map,
        "BsmtCond": qual_map,
        "HeatingQC": qual_map,
        "KitchenQual": qual_map,
        "FireplaceQu": qual_map,
        "GarageQual": qual_map,
        "GarageCond": qual_map,
        "PoolQC": qual_map,
        "BsmtExposure": exposure_map,
        "GarageFinish": finish_map,
        "BsmtFinType1": fin_type_map,
        "BsmtFinType2": fin_type_map,
        "Functional": functional_map,
        "PavedDrive": paved_map,
    }
    for c, mapping in ordinal_cols.items():
        if c in all_features.columns:
            all_features[c] = all_features[c].astype("object").map(mapping).fillna(0).astype(float)

    # One-hot encode remaining categoricals
    all_features = pd.get_dummies(all_features, columns=all_features.select_dtypes(include=["object"]).columns, drop_first=False)

    # Drop ID if present
    all_features = all_features.drop(columns=["Id"], errors="ignore")

    # Split back
    X_all = all_features.iloc[: len(train_features)].copy()
    X_test = all_features.iloc[len(train_features) :].copy()

    # Correlation filtering must match training
    if y is not None and corr_threshold is not None and corr_threshold > 0:
        corr = X_all.corrwith(y)
        keep = corr.index[corr.abs() >= corr_threshold]
        X_all = X_all[keep]
        X_test = X_test[keep]

    return X_all, y, X_test


corr_threshold = 0.01
X_all, y, X_test = preprocess_features(train_df, test_df, corr_threshold=corr_threshold)

print("Data Ready (with feature engineering)!")
print(f"Training features (X_all): {X_all.shape}")
print(f"Testing features (X_test): {X_test.shape}")
print(f"corr_threshold: {corr_threshold}")

Data Ready (with feature engineering)!
Training features (X_all): (1460, 224)
Testing features (X_test): (1459, 224)
corr_threshold: 0.01


## Final Prediction and Submission
- Using the loaded model I generate price predictions for the test houses. The results are formatted into the required Kaggle structure and exported as submission.csv. This file represents the final output of the entire MLOps pipeline.

In [55]:
# Load Model from Registry and Generate Predictions

# Define the model name and version
model_name = "XGBoost"
model_version = "latest"
model_uri = f"models:/{model_name}/{model_version}"

print(f"Loading model from Registry: {model_uri}...")

# Load via pyfunc (Universal format - works for any registered model flavor)
loaded_model = mlflow.pyfunc.load_model(model_uri)

# Generate predictions (NO .fit() — model is already trained)
predictions = loaded_model.predict(X_test)

# Sanity check: confirm prediction shape and sample values
print(f"Predictions generated: {len(predictions)} rows")
print(f"Sample predictions (first 5): {predictions[:5].astype(int).tolist()}")

# Format into Kaggle submission structure
submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": predictions
})

# Export
submission.to_csv('submission.csv', index=False)

print("-" * 40)
print("SUCCESS: submission.csv is ready for Kaggle upload.")
print(submission.head())


Loading model from Registry: models:/XGBoost/latest...
Predictions generated: 1459 rows
Sample predictions (first 5): [124329, 162754, 188769, 191690, 192591]
----------------------------------------
SUCCESS: submission.csv is ready for Kaggle upload.
     Id      SalePrice
0  1461  124329.101562
1  1462  162754.843750
2  1463  188769.437500
3  1464  191690.312500
4  1465  192591.109375
